In [1]:
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

project_id = "mi-proyecto-net-510916"
dataset_id = "tu_empresa"
credentials_path = "../../credentials/service-account.json"

credentials = service_account.Credentials.from_service_account_file(credentials_path)
client = bigquery.Client(credentials=credentials, project=project_id)

table_ref_base = f"{project_id}.{dataset_id}"

queries = {
    "1. Ventas y Facturación Total por Categoría de Producto": f"""
        SELECT 
            c.name AS category_name,
            COUNT(DISTINCT oi.order_id) AS total_orders,
            SUM(oi.quantity) AS units_sold,
            ROUND(SUM((oi.unit_price * oi.quantity) * (1 - oi.discount)), 2) AS total_revenue
        FROM `{table_ref_base}.order_items` oi
        JOIN `{table_ref_base}.products` p ON oi.product_id = p.product_id
        JOIN `{table_ref_base}.categories` c ON p.category_id = c.category_id
        GROUP BY c.name
        ORDER BY total_revenue DESC;
    """,
    
    "2. Top 5 Clientes con Mayor Volumen de Compras (Ticket Global)": f"""
        SELECT 
            cust.customer_id,
            CONCAT(cust.first_name, ' ', cust.last_name) AS customer_name,
            cust.country,
            COUNT(o.order_id) AS total_orders,
            ROUND(SUM(p.amount), 2) AS lifetime_value
        FROM `{table_ref_base}.customers` cust
        JOIN `{table_ref_base}.orders` o ON cust.customer_id = o.customer_id
        JOIN `{table_ref_base}.payments` p ON o.order_id = p.order_id
        WHERE p.status = 'completed'
        GROUP BY cust.customer_id, customer_name, cust.country
        ORDER BY lifetime_value DESC
        LIMIT 5;
    """,

    "3. Rendimiento de Canales de Adquisición de Clientes": f"""
        SELECT 
            cust.acquisition_channel,
            COUNT(DISTINCT cust.customer_id) AS total_customers,
            COUNT(DISTINCT o.order_id) AS total_orders,
            ROUND(SUM(pay.amount), 2) AS total_revenue_generated
        FROM `{table_ref_base}.customers` cust
        LEFT JOIN `{table_ref_base}.orders` o ON cust.customer_id = o.customer_id
        LEFT JOIN `{table_ref_base}.payments` pay ON o.order_id = pay.order_id AND pay.status = 'completed'
        GROUP BY cust.acquisition_channel
        ORDER BY total_revenue_generated DESC;
    """,

    "4. Estado de los Pedidos y Métodos de Pago más Utilizados": f"""
        SELECT 
            pay.payment_method,
            o.status AS order_status,
            COUNT(o.order_id) AS quantity_orders,
            ROUND(SUM(pay.amount), 2) AS total_amount
        FROM `{table_ref_base}.orders` o
        JOIN `{table_ref_base}.payments` pay ON o.order_id = pay.order_id
        GROUP BY pay.payment_method, o.status
        ORDER BY quantity_orders DESC;
    """,

    "5. Productos Peor Valorados (Media de Estrellas en Reviews)": f"""
        SELECT 
            p.product_id,
            p.name AS product_name,
            COUNT(r.review_id) AS total_reviews,
            ROUND(AVG(r.rating), 2) AS average_rating
        FROM `{table_ref_base}.reviews` r
        JOIN `{table_ref_base}.order_items` oi ON r.order_item_id = oi.order_item_id
        JOIN `{table_ref_base}.products` p ON oi.product_id = p.product_id
        GROUP BY p.product_id, product_name
        HAVING total_reviews >= 2
        ORDER BY average_rating ASC
        LIMIT 5;
    """
}

for title, query in queries.items():
    print("=" * 80)
    print(f"CONSULTA: {title}")
    print("=" * 80)
    df_result = client.query(query).to_dataframe()
    display(df_result.head(10))
    print("\n")

CONSULTA: 1. Ventas y Facturación Total por Categoría de Producto


c:\Users\y0rd4\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_name,total_orders,units_sold,total_revenue
0,Laptops,1987,55545,37111255.03
1,Smartphones,1973,57228,34528380.96
2,Audio,1964,48516,31449625.67
3,Periféricos,1957,47214,29751868.06
4,Wearables,1813,40005,27495006.53




CONSULTA: 2. Top 5 Clientes con Mayor Volumen de Compras (Ticket Global)


c:\Users\y0rd4\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


...

c:\Users\y0rd4\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,payment_method,order_status,quantity_orders,total_amount
0,paypal,shipped,1435,4198206.22
1,credit_card,shipped,1417,4189984.51
2,klarna,delivered,1416,4262345.96
3,paypal,delivered,1391,4201701.36
4,klarna,shipped,1387,4218237.39
5,credit_card,delivered,1344,3828118.77
6,apple_pay,shipped,1314,3806444.16
7,apple_pay,delivered,1255,3807621.87
8,klarna,confirmed,902,2712039.37
9,paypal,confirmed,884,2592086.77




CONSULTA: 5. Productos Peor Valorados (Media de Estrellas en Reviews)


c:\Users\y0rd4\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_id,product_name,total_reviews,average_rating
0,PROD_039,Tech Device 39,186,4.13
1,PROD_028,Tech Device 28,192,4.22
2,PROD_015,Tech Device 15,210,4.23
3,PROD_050,Tech Device 50,165,4.25
4,PROD_058,Tech Device 58,156,4.25
